# WS-04 — Tạo mẫu CSV gán nhãn yếu v2

Notebook này **chỉ tạo file CSV mẫu** cho toàn bộ cặp $\text{ca bệnh} \times \text{mệnh đề}$.

- Không gọi LLM/OpenAI.
- Không chạy ConceptCLIP.
- Không tạo, sửa hoặc ghi đè artifact WS-02 trong `WeakSup/propositions/v2/`.
- Chỉ ghi file đầu ra vào `WeakSup/weak_labels/v2/llm_pilot/pilot.csv`.

Mỗi dòng CSV đại diện cho một cặp `case_id` và `proposition_id`. Ba cột cần gán nhãn thủ công/sau này là `verdict`, `quote`, `quote_valid` được để trống.

## 1. Xác định cấu trúc dữ liệu CSV

CSV đầu ra có đúng các cột tương thích với file pilot của WS-03:

`case_id`, `disease_id`, `caption_masked`, `proposition_id`, `canonicalDescription`, `category`, `verdict`, `quote`, `quote_valid`, `same_disease`.

`verdict`, `quote`, `quote_valid` được giữ trống. `same_disease` là `1` khi bệnh đã gán cho ca trùng bệnh của mệnh đề, ngược lại là `0`.

In [ ]:
from pathlib import Path

CSV_COLUMNS = [
    "case_id",
    "disease_id",
    "caption_masked",
    "proposition_id",
    "canonicalDescription",
    "category",
    "verdict",
    "quote",
    "quote_valid",
    "same_disease",
]

print("CSV columns:")
for column in CSV_COLUMNS:
    print(f"- {column}")

## 2. Tạo dữ liệu mẫu

Kết nối Google Drive, lấy branch `WS`, đọc cấu hình SkinCAP và artifact WS-02 v2. Notebook đọc caption từ ba split, che alias bệnh bằng `disease_alias_v2.csv`, rồi tạo mọi tổ hợp giữa ca bệnh và mệnh đề.

> `disease_alias_v2.csv` phải đã được duyệt tại `WeakSup/weak_labels/v2/`. Notebook dừng rõ ràng nếu file này chưa tồn tại để tránh tạo caption che bệnh không được kiểm tra.

In [ ]:
import subprocess
import sys

try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError("Hãy chạy notebook này trong Google Colab.") from error

drive.mount("/content/drive", force_remount=False)

REPO_URL = "https://github.com/trisle0805/RAD-colab.git"
REPO_DIR = Path("/content/RAD-colab")
BRANCH = "WS"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

current_branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"],
    text=True,
).strip()
if current_branch != BRANCH:
    raise RuntimeError(f"Expected branch {BRANCH!r}, got {current_branch!r}")

requirements_file = REPO_DIR / "requirements" / "colab-skin.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Missing Colab requirements: {requirements_file}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", str(requirements_file)],
    check=True,
)
print(f"Repository ready: {REPO_DIR} ({current_branch})")

In [ ]:
import pandas as pd
import yaml

sys.path.insert(0, str(REPO_DIR))
from proposition.weak.labels import load_propositions, mask_disease_aliases, read_alias_csv

CONFIG_PATH = REPO_DIR / "configs" / "ws.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

if Path(cfg["repo_dir"]) != REPO_DIR:
    raise ValueError(f"configs/ws.yaml repo_dir must be {REPO_DIR}, got {cfg['repo_dir']}")

WS_ROOT = Path(cfg["ws_root"])
PROPOSITIONS_DIR = WS_ROOT / "propositions" / "v2"  # WS-02: read-only
OUTPUT_DIR = WS_ROOT / "weak_labels" / "v2"
ALIAS_CSV = OUTPUT_DIR / "disease_alias_v2.csv"
PILOT_DIR = OUTPUT_DIR / "llm_pilot"
OUTPUT_CSV = PILOT_DIR / "pilot.csv"

INPUT_PATHS = {
    "train": Path(cfg["skincap"]["train_csv"]),
    "val": Path(cfg["skincap"]["val_csv"]),
    "test": Path(cfg["skincap"]["test_csv"]),
    "propositions_final": PROPOSITIONS_DIR / "propositions_final.json",
    "propositions_review": PROPOSITIONS_DIR / "propositions_review.csv",
    "alias": ALIAS_CSV,
}

for label, path in INPUT_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing required {label} input: {path}")

print("Read-only inputs:")
for label, path in INPUT_PATHS.items():
    print(f"- {label}: {path}")
print("CSV output:", OUTPUT_CSV)

In [ ]:
def load_cases() -> list[dict[str, str]]:
    cases: list[dict[str, str]] = []
    expected_disease_columns: list[str] | None = None

    for split in ("train", "val", "test"):
        frame = pd.read_csv(INPUT_PATHS[split], dtype=str).fillna("")
        if frame.empty or len(frame.columns) < 3:
            raise ValueError(f"Invalid SkinCAP {split} CSV: {INPUT_PATHS[split]}")

        case_column, caption_column = frame.columns[:2]
        disease_columns = list(frame.columns[2:])
        if expected_disease_columns is None:
            expected_disease_columns = disease_columns
        elif disease_columns != expected_disease_columns:
            raise ValueError(f"Disease columns in {split} do not match train CSV.")

        for _, item in frame.iterrows():
            positives = [
                disease_id
                for disease_id in disease_columns
                if str(item[disease_id]).strip() in {"1", "1.0", "True", "true"}
            ]
            if len(positives) != 1:
                raise ValueError(
                    f"Expected exactly one active disease for case {item[case_column]!r} "
                    f"in {split}; got {positives!r}."
                )
            cases.append(
                {
                    "case_id": str(item[case_column]),
                    "disease_id": positives[0],
                    "caption": str(item[caption_column]),
                    "split": split,
                }
            )

    case_ids = [case["case_id"] for case in cases]
    if len(case_ids) != len(set(case_ids)):
        raise ValueError("case_id must be unique across train, val, and test.")
    return cases

PROPOSITIONS = load_propositions(
    INPUT_PATHS["propositions_final"],
    INPUT_PATHS["propositions_review"],
)
if not PROPOSITIONS:
    raise ValueError("WS-02 propositions v2 is empty.")

ALIASES = read_alias_csv(ALIAS_CSV)
CASES = load_cases()

rows: list[dict[str, object]] = []
for case in CASES:
    caption_masked = mask_disease_aliases(case["caption"], ALIASES)
    for proposition in PROPOSITIONS:
        rows.append(
            {
                "case_id": case["case_id"],
                "disease_id": case["disease_id"],
                "caption_masked": caption_masked,
                "proposition_id": proposition.proposition_id,
                "canonicalDescription": proposition.canonical_description,
                "category": proposition.category,
                "verdict": "",
                "quote": "",
                "quote_valid": "",
                "same_disease": int(case["disease_id"] == proposition.disease_id),
            }
        )

label_template = pd.DataFrame(rows, columns=CSV_COLUMNS)
expected_rows = len(CASES) * len(PROPOSITIONS)
if len(label_template) != expected_rows:
    raise AssertionError(f"Expected {expected_rows} rows, created {len(label_template)}.")

print(f"Cases: {len(CASES):,}")
print(f"Propositions: {len(PROPOSITIONS):,}")
print(f"CSV rows (case × proposition): {len(label_template):,}")
display(label_template.head())

## 3. Ghi dữ liệu ra tệp CSV

Xuất mẫu CSV với mã hóa `utf-8-sig` để mở trực tiếp bằng Excel. Ba cột `verdict`, `quote`, `quote_valid` vẫn để trống và sẵn sàng cho bước gán nhãn.

In [ ]:
PILOT_DIR.mkdir(parents=True, exist_ok=True)
label_template.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Đã tạo CSV mẫu: {OUTPUT_CSV}")
print(f"Kích thước: {OUTPUT_CSV.stat().st_size:,} bytes")

## 4. Đọc lại và kiểm tra tệp CSV

Đọc lại file vừa ghi để xác nhận thứ tự cột, số dòng và ba cột nhãn còn trống.

In [ ]:
check_frame = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False, encoding="utf-8-sig")

if list(check_frame.columns) != CSV_COLUMNS:
    raise AssertionError("CSV column order/schema does not match the required pilot schema.")
if len(check_frame) != len(label_template):
    raise AssertionError("CSV row count changed after writing.")
for column in ("verdict", "quote", "quote_valid"):
    if check_frame[column].ne("").any():
        raise AssertionError(f"{column} must be blank in the generated template.")

print(f"Verified: {len(check_frame):,} rows and {len(check_frame.columns)} columns.")
print("Label columns are blank; no LLM or image model was called.")
display(check_frame.head())

## 5. Xuất CSV khảo sát theo từng bệnh

Ba ô sau tách riêng để phục vụ khảo sát prompt theo từng `disease_id`:

1. Tạo thư mục bệnh.
2. Xuất `all_cases.csv` cho từng bệnh.
3. Lấy lại ngẫu nhiên tối đa 5 ca/bệnh và ghi đè `sample_5_cases.csv`.

Ô thứ ba không đặt seed cố định. Đặt `SAMPLE_MODE = "same_disease_only"` để file mẫu chỉ giữ các dòng có `same_disease = 1`: mỗi caption chỉ được ghép với các mệnh đề thuộc đúng bệnh đó. Điều này giúp khảo sát điểm chung giữa caption và mệnh đề của **từng bệnh**. Có thể đổi sang `"all_propositions"` khi cần giữ mọi mệnh đề.

In [ ]:
# Ô 1: Tạo cấu trúc thư mục theo disease_id.
# Chạy một lần trước khi xuất các file CSV theo từng bệnh.
DISEASE_DIRS = {
    disease_id: PILOT_DIR / disease_id
    for disease_id in sorted({case["disease_id"] for case in CASES})
}

for disease_dir in DISEASE_DIRS.values():
    disease_dir.mkdir(parents=True, exist_ok=True)

print(f"Đã sẵn sàng {len(DISEASE_DIRS):,} thư mục bệnh trong: {PILOT_DIR}")
for disease_id, disease_dir in list(DISEASE_DIRS.items())[:10]:
    print(f"- {disease_id}: {disease_dir}")

In [ ]:
# Ô 2: Xuất toàn bộ ca × toàn bộ mệnh đề cho từng bệnh.
# File này bao gồm cả các ca sau đó có thể được chọn vào sample_5_cases.csv.
if "DISEASE_DIRS" not in globals():
    raise RuntimeError("Hãy chạy ô tạo cấu trúc thư mục trước.")

all_case_counts: dict[str, int] = {}
for disease_id, disease_dir in DISEASE_DIRS.items():
    disease_rows = label_template.loc[label_template["disease_id"] == disease_id]
    expected_rows = sum(case["disease_id"] == disease_id for case in CASES) * len(PROPOSITIONS)
    if len(disease_rows) != expected_rows:
        raise AssertionError(
            f"Unexpected row count for {disease_id}: expected {expected_rows}, got {len(disease_rows)}."
        )

    all_cases_path = disease_dir / "all_cases.csv"
    disease_rows.to_csv(all_cases_path, index=False, encoding="utf-8-sig")
    all_case_counts[disease_id] = len(disease_rows)

print(f"Đã ghi all_cases.csv cho {len(all_case_counts):,} bệnh.")
print(f"Tổng số dòng: {sum(all_case_counts.values()):,}")

In [ ]:
# Ô 3: Lấy ngẫu nhiên tối đa 5 ca cho mỗi bệnh rồi ghi đè sample_5_cases.csv.
# Không đặt seed: chạy lại riêng ô này để lấy một mẫu khảo sát khác.
# "same_disease_only" chỉ xuất caption × các mệnh đề của chính bệnh đó.
# Đổi thành "all_propositions" nếu cần caption × toàn bộ mệnh đề.
if "DISEASE_DIRS" not in globals():
    raise RuntimeError("Hãy chạy ô tạo cấu trúc thư mục trước.")

SAMPLE_CASES_PER_DISEASE = 5
SAMPLE_MODE = "same_disease_only"
if SAMPLE_MODE not in {"same_disease_only", "all_propositions"}:
    raise ValueError("SAMPLE_MODE must be 'same_disease_only' or 'all_propositions'.")

sample_summary: list[dict[str, object]] = []
for disease_id, disease_dir in DISEASE_DIRS.items():
    disease_case_ids = [case["case_id"] for case in CASES if case["disease_id"] == disease_id]
    sample_size = min(SAMPLE_CASES_PER_DISEASE, len(disease_case_ids))
    sampled_case_ids = pd.Series(disease_case_ids, dtype="string").sample(
        n=sample_size,
        replace=False,
    ).tolist()

    sampled_rows = label_template.loc[label_template["case_id"].isin(sampled_case_ids)].copy()
    if SAMPLE_MODE == "same_disease_only":
        sampled_rows = sampled_rows.loc[sampled_rows["same_disease"] == 1].copy()
        propositions_per_case = sum(prop.disease_id == disease_id for prop in PROPOSITIONS)
    else:
        propositions_per_case = len(PROPOSITIONS)

    expected_rows = sample_size * propositions_per_case
    if len(sampled_rows) != expected_rows:
        raise AssertionError(
            f"Unexpected sampled row count for {disease_id}: expected {expected_rows}, got {len(sampled_rows)}."
        )
    if SAMPLE_MODE == "same_disease_only" and sampled_rows["same_disease"].ne(1).any():
        raise AssertionError(f"sample_5_cases.csv for {disease_id} contains a different-disease proposition.")

    sample_path = disease_dir / "sample_5_cases.csv"
    sampled_rows.to_csv(sample_path, index=False, encoding="utf-8-sig")
    sample_summary.append(
        {
            "disease_id": disease_id,
            "mode": SAMPLE_MODE,
            "available_cases": len(disease_case_ids),
            "sampled_cases": sample_size,
            "propositions_per_sampled_case": propositions_per_case,
            "sampled_case_ids": ", ".join(sampled_case_ids),
            "csv_rows": len(sampled_rows),
            "path": str(sample_path),
        }
    )

sample_summary_frame = pd.DataFrame(sample_summary)
print(f"Đã ghi đè sample_5_cases.csv theo mode: {SAMPLE_MODE}")
display(sample_summary_frame)

In [ ]:
# Ô 4: Tính tổng dung lượng các file sample_5_cases.csv.
if "DISEASE_DIRS" not in globals():
    raise RuntimeError("Hãy chạy ô tạo cấu trúc thư mục trước.")

sample_files = [
    disease_dir / "sample_5_cases.csv"
    for disease_dir in DISEASE_DIRS.values()
    if (disease_dir / "sample_5_cases.csv").is_file()
]
missing_samples = len(DISEASE_DIRS) - len(sample_files)
if not sample_files:
    raise FileNotFoundError("Chưa tìm thấy sample_5_cases.csv. Hãy chạy ô tạo mẫu ngẫu nhiên trước.")

total_bytes = sum(path.stat().st_size for path in sample_files)
print(f"Files found: {len(sample_files):,}/{len(DISEASE_DIRS):,}")
print(f"Missing sample files: {missing_samples:,}")
print(f"Total size: {total_bytes:,} bytes ({total_bytes / 1024**2:.2f} MiB)")

In [ ]:
# Ô 5: Gom bản sao của toàn bộ sample_5_cases.csv vào một thư mục dễ mở trên Drive.
# Tiền tố "00_" giúp thư mục đứng đầu khi Drive sắp xếp theo tên.
import shutil

if "DISEASE_DIRS" not in globals():
    raise RuntimeError("Hãy chạy ô tạo cấu trúc thư mục trước.")

SAMPLES_COLLECTION_DIR = PILOT_DIR / "00_sample_5_cases_all_diseases"
SAMPLES_COLLECTION_DIR.mkdir(parents=True, exist_ok=True)

copied_files = []
missing_diseases = []
for disease_id, disease_dir in DISEASE_DIRS.items():
    source_path = disease_dir / "sample_5_cases.csv"
    if not source_path.is_file():
        missing_diseases.append(disease_id)
        continue

    destination_path = SAMPLES_COLLECTION_DIR / f"{disease_id}_sample_5_cases.csv"
    shutil.copy2(source_path, destination_path)
    copied_files.append(destination_path)

print(f"Thư mục tổng hợp: {SAMPLES_COLLECTION_DIR}")
print(f"Đã copy: {len(copied_files):,}/{len(DISEASE_DIRS):,} file sample.")
if missing_diseases:
    print(f"Chưa có sample_5_cases.csv cho {len(missing_diseases):,} bệnh: {missing_diseases[:10]}")